In [1]:
import base64
import re
import textwrap
from io import BytesIO, StringIO
from pathlib import Path
import numpy as np
from docling.document_converter import DocumentConverter, PdfFormatOption # for PDF content extraction
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TesseractCliOcrOptions, smolvlm_picture_description
from dotenv import load_dotenv
from IPython.display import HTML, display
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import json
from typing import List
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama.llms import OllamaLLM
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter


/opt/homebrew/Caskroom/miniconda/base/envs/detest/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def image_to_base64(img: Image.Image) -> str:
    buffered = BytesIO()
    img.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return img_str

In [3]:
def display_page(content:str, image:Image):
    html_template = f"""
    <div style="display: flex; asign-items: flex-start; gap: 40px; font-family: monospace;">
        <div style="flex: 1; max-width: 45%;">
            <img src="data:image/png;base64,{image_to_base64(image)}" style="max-width: 100%; height: auto; padding: 5px;"/>
        </div>
        <div style="flex: 1; max-width: 45%; white-space: pre-wrap; padding: 10px;">
            <div style="word-wrap: break-word; max-width: 120ch;">
            {content}
            </div>
        </div>
    </div>
    """

    display(HTML(html_template))

In [4]:
def extract_pdf_content(file_path, page_range) -> str:
    pipeline_options=PdfPipelineOptions(
                                        generate_page_images=True,
                                        images_scale=1.00,
                                        do_picture_description=True,
                                        picture_description_options=smolvlm_picture_description,
                                        do_ocr=True,
                                        ocr_options=TesseractCliOcrOptions(lang=["en"])
                                        )
    # pipeline_options.do_ocr = True
    # pipeline_options.ocr_options = TesseractCliOcrOptions(lang=["vie"])
    converter = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)})
    result = converter.convert(file_path,page_range= page_range)
    return result#.document.export_to_dict()

In [5]:
pdf_path = "/Users/Home/GitHub/Retrieve/AFP-GOCA-Reference-Graphics-Object-Content-Architecture-for-AFP-Reference.pdf"
pdf_content = extract_pdf_content(pdf_path,(102,181)).document
# num_pages = len(pdf_content.pages)
# pages=[]
# for page_num in range(0,num_pages+1):
#     pages.append(
#         (
#             pdf_content.export_to_markdown(page_no=page_num),
#             pdf_content.pages[page_num].image.pil_image
#         )
#     )
#display_page(*pages[1])

2025-10-15 22:51:13,691 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-15 22:51:13,721 - INFO - Going to convert document batch...
2025-10-15 22:51:13,722 - INFO - Initializing pipeline for StandardPdfPipeline with options hash a0d21e9190b43ad25d3af43855090692
2025-10-15 22:51:13,726 - INFO - Loading plugin 'docling_defaults'
2025-10-15 22:51:13,728 - INFO - Registered ocr engines: ['easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-10-15 22:51:13,760 - INFO - command: tesseract --list-langs
2025-10-15 22:51:13,795 - INFO - Accelerator device: 'mps'
2025-10-15 22:51:15,086 - INFO - Accelerator device: 'mps'
2025-10-15 22:51:15,441 - INFO - Loading plugin 'docling_defaults'
2025-10-15 22:51:15,442 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-10-15 22:51:15,730 - INFO - Accelerator device: 'mps'
`torch_dtype` is deprecated! Use `dtype` instead!
2025-10-15 22:51:17,219 - INFO - Processing document AFP-GOCA-Reference-Graphics-Object-Content-Arch

In [6]:
ex_content=pdf_content.export_to_markdown()
content=ex_content.__str__()
doc_content=content.split("\n\n")

In [7]:
def take_tables(doc_content,num_seq=6):
    tables =[]
    tables_content= [table for table in doc_content if "|" in table]
    for table_content in tables_content:
        content = table_content.split("\n")
        if content[0].count("|")==num_seq:
            tables.append(table_content)
    return tables


In [8]:
def clean_table(content):
    buffer = StringIO(content)
    table = pd.read_table(filepath_or_buffer=buffer,
                   sep="|",
                   lineterminator="\n",
                   header=0)
    table = table.iloc[:, 1:-1].iloc[1:]
    # Bo khoang trang
    table.columns = table.columns.str.strip()
    for col in table.select_dtypes(include="object").columns:
        table[col] = table[col].str.strip()
    return table

In [9]:
# Take list of codes
list_page = extract_pdf_content(pdf_path,(100,101)).document.export_to_markdown().__str__().split("\n\n")
list_tables = take_tables(list_page,num_seq=5)
lists=[]
for content in list_tables:
    list_table=clean_table(content)
    lists.append(list_table)
list_table = pd.concat([lists[0],lists[1]],ignore_index=True)
list_code = {}
for i in range(len(list_table)):
    list_code[list_table['Order'].iloc[i]] = list_table['Meaning'].iloc[i]

2025-10-15 22:52:35,184 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-15 22:52:35,193 - INFO - Going to convert document batch...
2025-10-15 22:52:35,194 - INFO - Initializing pipeline for StandardPdfPipeline with options hash a0d21e9190b43ad25d3af43855090692
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
2025-10-15 22:52:35,234 - INFO - command: tesseract --list-langs
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
2025-10-15 

In [10]:
def concat_tables(first_table, second_table):
    if first_table is None or getattr(first_table, "empty", True):
        return second_table
    else:
        startbit = second_table["Offset"].iloc[0].split(" ")[0]
        if startbit != "0":
            table = pd.concat([first_table,second_table],ignore_index=True)
            return table
        else:
            return second_table
        

In [11]:
def take_tag(table):
    tag = table["Meaning"].iloc[0].split(" ")[0]
    tagorder = re.findall(r"[A-Z0-9]+",tag)[0]
    return tagorder

In [17]:
def take_length(table):
    if len(table)<2:
        lengthorder = 1
    elif table["Name"].iloc[1]== "LENGTH":
        lengthorder = table["Range"].iloc[1]
    else:
        lengthorder = "1"
    return lengthorder

In [13]:
def take_code(table):
    code = table["Range"].iloc[0]
    match=re.search(rf"X'([a-zA-Z0-9]+)'",code)
    if match:
        codeorder = match.group(1)
    return codeorder

In [14]:
def take_description(tagorder,doc_content):
    ind = []
    for index, content in enumerate(doc_content):
        match = re.match(rf"##.*{tagorder}.*", content.__str__())
        if match!=None:
            ind.append(index)
    descriptionorder = doc_content[ind[0]+1]
    return descriptionorder

In [18]:
tables_content = take_tables(doc_content)
table =[]
info = {}
for content in tables_content:
    oldtable = table
    newtable = clean_table(content)
    table = concat_tables(oldtable,newtable)
    tagorder = take_tag(table)
    nameorder = list_code[tagorder]
    lengthorder = take_length(table)
    codeorder = take_code(table)
    descriptionorder = take_description(tagorder,doc_content)
    info[nameorder] = {"Tag": tagorder, "Code": codeorder, "Length": lengthorder, "Description": descriptionorder}

In [19]:
output_filename = f"output/table.json"

with open(output_filename,"w") as f:
    json.dump(info,f, indent=4, ensure_ascii=False)  
